Eda notebook 
- to understand the data some prior to update the dataset
- get an idea on how many rows will be dropped and nulls in the data

In [1]:
import pandas as pd 
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression

In [2]:
big_nss = pd.read_csv("../data/big_nss.csv")
big_nss.head()

C:\Users\blond\AppData\Local\Temp\ipykernel_72352\3568697817.py:1: DtypeWarning: Columns (3) have mixed types. Specify dtype option on import or set low_memory=False.
  big_nss = pd.read_csv("../data/big_nss.csv")


,Unnamed: 0,RecordID,EventTimeStamp,EquipmentID,EngineOilPressure,EngineOilTemperature,DistanceLtd,FuelLtd,EngineTimeLtd,Derate
0,0,1211417,2000-03-18 19:14:10,2015,25.52,190.85,274765.4,37866.421934,5673.1,False
1,1,1211418,2000-03-18 19:14:10,2015,25.52,190.85,274765.4,37866.421934,5673.1,False
2,2,1211419,2000-03-18 19:20:47,2015,NaN,NaN,NaN,NaN,NaN,False
3,3,1211420,2000-03-18 19:20:47,2015,NaN,NaN,NaN,NaN,NaN,False
4,4,1211422,2000-03-19 02:59:58,1849,NaN,NaN,NaN,NaN,NaN,False


In [3]:
big_nss.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1187335 entries, 0 to 1187334
Data columns (total 10 columns):
 #   Column                Non-Null Count    Dtype  
---  ------                --------------    -----  
 0   Unnamed: 0            1187335 non-null  int64  
 1   RecordID              1187335 non-null  int64  
 2   EventTimeStamp        1187335 non-null  object 
 3   EquipmentID           1187335 non-null  object 
 4   EngineOilPressure     586244 non-null   float64
 5   EngineOilTemperature  583912 non-null   float64
 6   DistanceLtd           585819 non-null   float64
 7   FuelLtd               585195 non-null   float64
 8   EngineTimeLtd         581366 non-null   float64
 9   Derate                1187335 non-null  bool   
dtypes: bool(1), float64(5), int64(2), object(2)
memory usage: 82.7+ MB


In [4]:
# null counts per column before any changes
print(f"Total rows: {len(big_nss)}")
print("\nNull counts per column:")
big_nss.isna().sum()

Total rows: 1187335

Null counts per column:


Unnamed: 0                   0
RecordID                     0
EventTimeStamp               0
EquipmentID                  0
EngineOilPressure       601091
EngineOilTemperature    603423
DistanceLtd             601516
FuelLtd                 602140
EngineTimeLtd           605969
Derate                       0
dtype: int64

In [5]:
# rows where both oil columns AND all 3 ltd columns are null
oil_cols = ['EngineOilPressure', 'EngineOilTemperature']
ltd_cols = ['DistanceLtd', 'FuelLtd', 'EngineTimeLtd']

mask = big_nss[oil_cols].isna().all(axis=1) & big_nss[ltd_cols].isna().all(axis=1)
print(f"Rows to be dropped: {mask.sum()}")


Rows to be dropped: 599700


In [6]:

all_null_per_equipment = big_nss.groupby('EquipmentID')[ltd_cols].apply(lambda g: g.isna().all())
bad_ids = all_null_per_equipment[all_null_per_equipment.sum(axis=1) >= 2].index

print(f"Equipment IDs to drop: {len(bad_ids)}")
print(f"Rows to drop: {big_nss[big_nss['EquipmentID'].isin(bad_ids)].shape[0]}")

Equipment IDs to drop: 33
Rows to drop: 73


In [7]:
# checking on non altered dataset since the check i had said there were no rows updating and it seemed odd

has_some = big_nss[ltd_cols].notna().any(axis=1)
has_missing = big_nss[ltd_cols].isna().any(axis=1)

print(f"Rows with at least 1 Ltd value but missing others: {(has_some & has_missing).sum()}")


Rows with at least 1 Ltd value but missing others: 8652


In [9]:
with pd.option_context('display.max_rows', None):
    display(big_nss[big_nss['EquipmentID'] == 301])

,Unnamed: 0,RecordID,EventTimeStamp,EquipmentID,EngineOilPressure,EngineOilTemperature,DistanceLtd,FuelLtd,EngineTimeLtd,Derate
66977,66977,68495,2015-05-28 08:07:38,301,NaN,NaN,NaN,NaN,NaN,False
67432,67432,68953,2015-05-28 13:31:41,301,51.620000,215.206300,125989.2,17776.401551,2475.90,False
67566,67566,69086,2015-05-28 15:12:40,301,NaN,NaN,NaN,NaN,NaN,False
69830,69830,71663,2015-05-31 05:27:55,301,NaN,NaN,NaN,NaN,NaN,False
70009,70009,71844,2015-05-31 11:48:43,301,NaN,NaN,NaN,NaN,NaN,False
70010,70010,71845,2015-05-31 11:48:45,301,NaN,NaN,NaN,NaN,NaN,False
75861,75861,77719,2015-06-05 05:39:28,301,NaN,NaN,NaN,NaN,NaN,False
79600,79600,81434,2015-06-09 05:56:20,301,NaN,NaN,NaN,NaN,NaN,False
79615,79615,81446,2015-06-09 06:05:49,301,NaN,NaN,NaN,NaN,NaN,False
81283,81283,83119,2015-06-10 11:55:42,301,53.940000,201.706300,129342.4,18232.758771,2556.90,False


In [8]:
print(big_nss[(big_nss['EquipmentID'] == '301')])

         Unnamed: 0  RecordID       EventTimeStamp EquipmentID  \
1108           1108       561  2015-02-22 05:03:32         301   
1109           1109       560  2015-02-22 05:03:32         301   
1110           1110       559  2015-02-22 05:03:32         301   
18133         18133     18465  2015-04-10 15:53:35         301   
29649         29649     30246  2015-04-22 05:14:49         301   
...             ...       ...                  ...         ...   
1047520     1047520   1089011  2018-12-03 17:05:12         301   
1047523     1047523   1089014  2018-12-03 17:08:54         301   
1047524     1047524   1089013  2018-12-03 17:08:54         301   
1047525     1047525   1088988  2018-12-03 17:08:54         301   
1047526     1047526   1088987  2018-12-03 17:08:54         301   

         EngineOilPressure  EngineOilTemperature  DistanceLtd       FuelLtd  \
1108                   NaN                   NaN          NaN           NaN   
1109                   NaN                   NaN 